In [ ]:
# Get the income summary stats

import geopandas as gpd
from pathlib import Path
import pandas as pd
import requests
import openpyxl

In [ ]:
# Set up paths

current_path = Path.cwd()

path_root = current_path.parent.parent

data_path = path_root / "data" / "forecasting"
file_path = data_path / "IncomeSummaryStats_1823.xlsx"

# Create the data folder if it does not exist
data_path.mkdir(parents=True, exist_ok=True)

# Download the 2018-19 to 2022-23 data
url = (
    "https://www.abs.gov.au/statistics/labour/"
    "earnings-and-working-conditions/personal-income-australia/"
    "2022-23/Table%201%20-%20Total%20income%2C%20earners"
    "%20and%20summary%20statistics%20by%20geography%2C%202018-"
    "19%20to%202022-23.xlsx"
)

response = requests.get(
    url,
    timeout=60,
)

# Stop if the download failed
response.raise_for_status()

# Save the raw Excel file
with open(file_path, "wb") as file:
    file.write(response.content)


print(f"Downloaded successfully to: {file_path}")

In [ ]:
# Convert the Excel sheet to a csv file

uncleaned_data = pd.read_excel(file_path,
                               sheet_name="Table 1.4",
                               header=None
                               )
csv1_file_path = data_path / "IncomeSummaryStats_1823_Uncleaned.csv"

uncleaned_data.to_csv(csv1_file_path)

In [ ]:
# Read the uncleaned csv back in and preview it

uncleaned_data = pd.read_csv(csv1_file_path)
uncleaned_data.head(10)

In [ ]:
# Clean the data, rename the columns and keep only Victorian SA2 codes

# Observations begin at row 7, row 6 holds the financial year labels
income_data = uncleaned_data.iloc[7:].copy()

# Remove the unnecessary first column
income_data = income_data.drop(
    columns=["Unnamed: 0"]
)

# Keep only rows with a 9 digit SA2 code
income_data = income_data[
    income_data["0"]
    .astype(str)
    .str.fullmatch(r"\d{9}")
].copy()

# Create the column names

years = [
    "2018_19",
    "2019_20",
    "2020_21",
    "2021_22",
    "2022_23",
]

column_names = [
    "sa2_code",
    "sa2_name",
]

column_names += [
    f"earners_{year}"
    for year in years
]

column_names += [
    f"median_age_{year}"
    for year in years
]

column_names += [
    f"total_income_{year}"
    for year in years
]

column_names += [
    f"median_income_{year}"
    for year in years
]

column_names += [
    f"mean_income_{year}"
    for year in years
]

income_data.columns = column_names

# Clean the data types

# Keep SA2 codes as strings since they are identifiers
income_data["sa2_code"] = (
    income_data["sa2_code"]
    .astype(str)
)


# Convert all income and statistic columns to numeric
numeric_columns = income_data.columns[
    2:
]

income_data[numeric_columns] = (
    income_data[numeric_columns]
    .apply(
        pd.to_numeric,
        errors="coerce",
    )
)

income_data = income_data.reset_index(
    drop=True
)


In [ ]:
# Keep Victorian SA2 codes, median age, median income and mean income

# Victorian SA2 codes start with 2
income_data = income_data[
    income_data["sa2_code"].str.startswith("2")
].copy()

income_data = income_data.reset_index(
    drop=True
)

income_data = income_data[
    [
        column
        for column in income_data.columns
        if (
            column in ["sa2_code", "sa2_name"]
            or "median_age" in column
            or "median_income" in column
            or "mean_income" in column
        )
    ]
].copy()

income_data.head()

In [ ]:
# Write the cleaned data to a csv file

csv2_file_path = data_path / "IncomeSummaryStats_1823_Cleaned.csv"
income_data.to_csv(csv2_file_path)

In [ ]:
# Download the older income datasets to extend the time series


url_1117 = ("https://www.abs.gov.au/statistics/labour/"
            "earnings-and-working-conditions/personal-"
            "income-australia/2011-12-2016-17/"
            "6124055002ds0001_2019.xls"
)

url_1419 = ("https://www.abs.gov.au/statistics/labour/"
            "earnings-and-working-conditions/personal-"
            "income-australia/2014-15-2018-19/"
            "6524055002_DO001.xlsx"
)


file_path_1117 = data_path / "IncomeSummaryStats_1117.xls"

file_path_1419 = data_path / "IncomeSummaryStats_1419.xlsx"


response_1117 = requests.get(
    url_1117,
    timeout=60
)

response_1419 = requests.get(
    url_1419,
    timeout=60
)


# Save both raw files
with open(file_path_1117, "wb") as file:
    file.write(response_1117.content)

with open(file_path_1419, "wb") as file:
    file.write(response_1419.content)

print(f"Downloaded successfully to: {file_path_1117}")
print(f"Downloaded successfully to: {file_path_1419}")

In [ ]:
# Read the Excel sheets for the older datasets

unclean_data_1117 = pd.read_excel(file_path_1117,
                                  sheet_name="Table 1.4",
                                  header=None
)

unclean_data_1419 = pd.read_excel(file_path_1419,
                                  sheet_name="Table 1.4",
                                  header=None)


In [ ]:
# Save the older datasets as unclean csv files

csv_1117_file_path = data_path / "IncomeSummaryStats_1117_Unclean.csv"
csv_1419_file_path = data_path / "IncomeSummaryStats_1419_Unclean.csv"

unclean_data_1117.to_csv(csv_1117_file_path)
unclean_data_1419.to_csv(csv_1419_file_path)

In [ ]:
# Read the unclean csv files back in to build one dataframe covering over 10 years

income_data_1117 = pd.read_csv(csv_1117_file_path)
income_data_1419 = pd.read_csv(csv_1419_file_path)

In [ ]:
# Reshape the 2011-12 to 2016-17 data to match the 2018-23 data

income_data_1117 = income_data_1117[
    [
        # SA2 information
        "0",
        "1",

        # Median age
        "8",
        "9",
        "10",
        "11",
        "12",
        "13",

        # Median income
        "20",
        "21",
        "22",
        "23",
        "24",
        "25",

        # Mean income
        "26",
        "27",
        "28",
        "29",
        "30",
        "31",
    ]
].copy()


years = [
    "2011_12",
    "2012_13",
    "2013_14",
    "2014_15",
    "2015_16",
    "2016_17",
]

column_names = [
    "sa2_code",
    "sa2_name",
]

column_names += [
    f"median_age_{year}"
    for year in years
]

column_names += [
    f"median_income_{year}"
    for year in years
]

column_names += [
    f"mean_income_{year}"
    for year in years
]

income_data_1117.columns = column_names

# Keep only rows with a 9 digit SA2 code
income_data_1117 = income_data_1117[
    income_data_1117["sa2_code"]
    .astype(str)
    .str.fullmatch(r"\d{9}")
].copy()

income_data_1117["sa2_code"] = (
    income_data_1117["sa2_code"]
    .astype(str)
)

# Keep only Victorian SA2 codes
income_data_1117 = income_data_1117[
    income_data_1117["sa2_code"]
    .str.startswith("2")
].copy()

numeric_columns = (
    income_data_1117.columns[2:]
)

income_data_1117[numeric_columns] = (
    income_data_1117[numeric_columns]
    .apply(
        pd.to_numeric,
        errors="coerce",
    )
)

income_data_1117 = (
    income_data_1117
    .reset_index(drop=True)
)

In [ ]:
# Extract the 2017-18 data

income_data_1718 = income_data_1419[
    [
        "0",   # SA2 code
        "1",   # SA2 name
        "10",  # Median age 2017-18
        "20",  # Median income 2017-18
        "25",  # Mean income 2017-18
    ]
].copy()

income_data_1718.columns = [
    "sa2_code",
    "sa2_name",
    "median_age_2017_18",
    "median_income_2017_18",
    "mean_income_2017_18",
]

# Keep only rows with a 9 digit SA2 code
income_data_1718 = income_data_1718[
    income_data_1718["sa2_code"]
    .astype(str)
    .str.fullmatch(r"\d{9}")
].copy()

income_data_1718["sa2_code"] = (
    income_data_1718["sa2_code"]
    .astype(str)
)

# Keep only Victorian SA2 codes
income_data_1718 = income_data_1718[
    income_data_1718["sa2_code"]
    .str.startswith("2")
].copy()

numeric_columns = [
    "median_age_2017_18",
    "median_income_2017_18",
    "mean_income_2017_18",
]

income_data_1718[numeric_columns] = (
    income_data_1718[numeric_columns]
    .apply(
        pd.to_numeric,
        errors="coerce",
    )
)

income_data_1718 = (
    income_data_1718
    .reset_index(drop=True)
)


In [ ]:
# Read the cleaned 2018-23 data

income_data_1823 = pd.read_csv(csv2_file_path)

In [ ]:
# Make sure the SA2 code types match before merging

income_data_1117["sa2_code"] = (
    income_data_1117["sa2_code"]
    .astype(str)
)

income_data_1718["sa2_code"] = (
    income_data_1718["sa2_code"]
    .astype(str)
)

income_data_1823["sa2_code"] = (
    income_data_1823["sa2_code"]
    .astype(str)
)

In [ ]:
# Merge the three datasets on SA2 code

income_data = income_data_1117.merge(
    income_data_1718.drop(
        columns=["sa2_name"],
        errors="ignore",
    ),
    on="sa2_code",
    how="outer",
)

income_data = income_data.merge(
    income_data_1823.drop(
        columns=["sa2_name"],
        errors="ignore",
    ),
    on="sa2_code",
    how="outer",
)

In [ ]:
# Reorder the columns by measure and year

median_age_columns = sorted(
    [
        column
        for column in income_data.columns
        if column.startswith("median_age_")
    ]
)

median_income_columns = sorted(
    [
        column
        for column in income_data.columns
        if column.startswith("median_income_")
    ]
)

mean_income_columns = sorted(
    [
        column
        for column in income_data.columns
        if column.startswith("mean_income_")
    ]
)

income_data = income_data[
    [
        "sa2_code",
        "sa2_name",
    ]
    + median_age_columns
    + median_income_columns
    + mean_income_columns
].copy()

In [ ]:
# Write the final merged data to a csv file

final_file_path = data_path / "IncomeSummaryStats_1123.csv"

income_data.to_csv(final_file_path)